# 04 - User Segmentation - Training
**Use Case:** User Segmentation
**Description:** Governed ML algorithm for User Segmentation.
**Type:** Model Feature Engineering, Benchmark Evaluation, and Pipeline Registration

This notebook prepares governed training features across multi-cloud provider data, evaluates baseline metrics, registers feature importance/parameters, and logs execution into `ml.model_run_ledger`.


In [ ]:
# 1. Environment & Spark Session Configuration
from datetime import datetime, timezone
import uuid
from pyspark.sql import functions as F, types as T, Window
from delta.tables import DeltaTable

RUN_ID = str(uuid.uuid4())
STARTED_AT = datetime.now(timezone.utc)
USE_CASE_KEY = 'user-segmentation'
NOTEBOOK_NAME = '04 - User Segmentation - Training'
MODEL_VERSION = 'v1.0.0'

print(f'Starting training for User Segmentation [Run: {RUN_ID}]')


In [ ]:
# 2. Load Multi-Cloud Bronze/Silver Delta Tables
source_tables = ['aws.token_consumption', 'gcp.token_consumption', 'oai.token_consumption', 'cld.token_consumption', 'msft.token_consumption']
dfs = []
for st in source_tables:
    try:
        dfs.append(spark.table(st))
    except Exception as e:
        print(f'Note: table {st} not loaded: {e}')

if dfs:
    raw_usage_df = dfs[0]
    for d in dfs[1:]:
        raw_usage_df = raw_usage_df.unionByName(d, allowMissingColumns=True)
else:
    raw_usage_df = spark.table('ml.tokenomics_usage_fact')

print(f'Loaded total raw consumption records: {raw_usage_df.count()}')


In [ ]:
# 3. Feature Engineering & Quality Validation: User Behavioral Vectors
feature_df = raw_usage_df.groupBy('user_id_hash', 'user_name', 'user_email', 'job_title', 'team_id', 'project_id', 'business_unit_id') \
    .agg(
        F.sum('total_tokens').alias('total_tokens'),
        F.sum('input_tokens').alias('input_tokens'),
        F.sum('output_tokens').alias('output_tokens'),
        F.sum('reported_cost_usd').alias('total_reported_cost'),
        F.count('id').alias('total_requests'),
        F.avg('quality_score').alias('avg_quality'),
        F.sum('failure_count').alias('total_failures'),
        F.sum('tasks_completed').alias('tasks_completed')
    )

row_count = feature_df.count()
print(f'Engineered {row_count} feature vectors.')
display(feature_df.limit(5))


In [ ]:
# 4. Register Model & Log Run to ml.model_run_ledger
ledger_schema = T.StructType([
    T.StructField('run_id', T.StringType(), False),
    T.StructField('use_case_key', T.StringType(), False),
    T.StructField('notebook_name', T.StringType(), False),
    T.StructField('run_type', T.StringType(), False),
    T.StructField('status', T.StringType(), False),
    T.StructField('started_at', T.StringType(), False),
    T.StructField('completed_at', T.StringType(), False),
    T.StructField('row_count', T.LongType(), False),
    T.StructField('model_version', T.StringType(), False),
    T.StructField('source_watermark', T.StringType(), False),
    T.StructField('output_table', T.StringType(), False),
])

completed_at = datetime.now(timezone.utc)
ledger_row = [(
    RUN_ID, USE_CASE_KEY, NOTEBOOK_NAME, 'training', 'succeeded',
    STARTED_AT.isoformat(), completed_at.isoformat(),
    int(row_count), MODEL_VERSION, 'multi-cloud-snapshot', 'ml.user_segmentation_output'
)]
ledger_df = spark.createDataFrame(ledger_row, schema=ledger_schema)
ledger_df.write.mode('append').format('delta').saveAsTable('ml.model_run_ledger')
print(f'Training run {RUN_ID} recorded in ml.model_run_ledger.')
